# UPI Transaction Analytics: Data Audit

## Project
Can UPI Transaction Data Predict India's Consumer Economy?

## Objective
The purpose of this notebook is to perform an initial audit of the UPI transaction dataset, assess data quality, identify issues, and prepare the data for further analysis.

## Audit Scope

This notebook covers:

- Data import and consolidation
- Dataset structure review
- Data type validation
- Missing value analysis
- Duplicate detection
- Data quality issue identification
- Data cleaning and standardization
- Data dictionary creation
- Audit summary

## Dataset

Source: NPCI UPI Statistics

Time Period: April 2016 – May 2026

Granularity: Monthly

## Expected Deliverable

A validated and cleaned dataset ready for exploratory data analysis (EDA).

# 1. Data Import

In [1]:
import pandas as pd

df = pd.read_excel("../data/raw/Product-Statistics-UPI-Product-statistics-upi-2024-25-Monthly.xlsx")

df.tail(n=15)

,Month,No. of Banks live on UPI,Volume (In Mn.),Value (In Cr.)
0,March-2025,661,"18,301.51\t","24,77,221.61"
1,February-2025,653,16106.19,"21,96,481.69"
2,January-2025,647,16996,"23,48,037.12"
3,December-2024,641,16730.01,"23,24,699.91"
4,November-2024,637,15482.02,"21,55,187.40"
5,October-2024,632,16584.97,"23,49,821.46"
6,September-2024,622,15041.75,"20,63,994.71"
7,August-2024,608,14963.05,"20,60,735.57"
8,July-2024,605,14435.55,"20,64,292.41"
9,June-2024,602,13885.14,"20,07,081.20"


In [2]:
df.columns

Index(['Month', 'No. of Banks live on UPI', 'Volume (In Mn.)',
       'Value (In Cr.)'],
      dtype='object')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Month                     12 non-null     object
 1   No. of Banks live on UPI  12 non-null     int64 
 2   Volume (In Mn.)           12 non-null     object
 3   Value (In Cr.)            12 non-null     object
dtypes: int64(1), object(3)
memory usage: 512.0+ bytes


In [4]:
import glob

files = glob.glob("../data/raw/*.xlsx")

dfs = []

for file in files:
    df = pd.read_excel(file)
    dfs.append(df)

upi = pd.concat(dfs, ignore_index=True)

upi.iloc[90:]

,Month,No. of Banks live on UPI,Volume (In Mn.),Value (In Cr.)
90,September-2023,492,10555.69,"15,79,133.18"
91,August-2023,484,10586.02,"15,76,536.56"
92,July-2023,473,9964.61,"15,33,536.44"
93,June-2023,458,9335.06,"14,75,464.26"
94,May-2023,445,9415.19,"14,89,145.44"
95,April-2023,414,8863.26,"14,15,504.71"
96,March-2025,661,"18,301.51\t","24,77,221.61"
97,February-2025,653,16106.19,"21,96,481.69"
98,January-2025,647,16996,"23,48,037.12"
99,December-2024,641,16730.01,"23,24,699.91"


In [5]:
upi = upi.rename(columns={
    'No. of Banks live on UPI': 'Banks',
    'Volume (In Mn.)': 'Volume_Mn',
    'Value (In Cr.)': 'Value_Cr'
})

# 2. Initial Dataset Inspection

## Data Quality Issue #1: Chronological Order

### Observation
After merging multiple yearly UPI files, the records were not in chronological order.

Example:

September-2023
August-2023
July-2023

followed by

March-2025
February-2025
January-2025

and then

March-2026
February-2026
January-2026

### Root Cause
Each yearly file was stored in reverse chronological order (latest month first). When the files were merged, the combined dataset inherited this ordering.

### Impact
Time-series calculations such as:

- Growth Rate (pct_change)
- Rolling Averages
- Trend Analysis
- Forecasting

would produce incorrect results if the dataset remains unsorted.

### Resolution
Convert the Month column to datetime format and sort the dataset in ascending chronological order.

### Status
Resolved

In [6]:
# Convert Month column to datetime
upi['Month'] = pd.to_datetime(upi['Month'])

# Sort chronologically
upi = upi.sort_values('Month')

#reset index
upi = upi.reset_index(drop='true')

In [7]:
upi.tail(n=25)

,Month,Banks,Volume_Mn,Value_Cr
97,2024-05-01,598,14035.84,"20,44,937.05"
98,2024-06-01,602,13885.14,"20,07,081.20"
99,2024-07-01,605,14435.55,"20,64,292.41"
100,2024-08-01,608,14963.05,"20,60,735.57"
101,2024-09-01,622,15041.75,"20,63,994.71"
102,2024-10-01,632,16584.97,"23,49,821.46"
103,2024-11-01,637,15482.02,"21,55,187.40"
104,2024-12-01,641,16730.01,"23,24,699.91"
105,2025-01-01,647,16996,"23,48,037.12"
106,2025-02-01,653,16106.19,"21,96,481.69"


# Initial Data Audit

In [8]:
upi.shape

(122, 4)

In [9]:
upi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Month      122 non-null    datetime64[ns]
 1   Banks      122 non-null    int64         
 2   Volume_Mn  122 non-null    object        
 3   Value_Cr   122 non-null    object        
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 3.9+ KB


In [10]:
upi.columns

Index(['Month', 'Banks', 'Volume_Mn', 'Value_Cr'], dtype='object')

In [11]:
upi.isna().sum()

Month        0
Banks        0
Volume_Mn    0
Value_Cr     0
dtype: int64

In [12]:
upi.describe()

,Banks
count,122.000000
mean,303.754098
std,229.654369
min,21.000000
25%,128.000000
50%,222.000000
75%,513.250000
max,720.000000


## Data Quality Issue #2: Numeric Columns Stored as Text

### Observation
The Volume and Value columns were imported as object data types instead of numeric.

### Root Cause
The source files contain formatting characters such as commas used as thousands separators.

Examples:

18,395.01

24,03,930.69

17,893.42

### Impact
Numeric calculations such as:

- Growth Rate Analysis
- Aggregations
- Correlation Analysis
- Forecasting

cannot be performed correctly.

### Resolution
Remove formatting characters and convert columns to numeric data types.

### Status
Resolved

In [13]:
upi[['Volume_Mn','Value_Cr']].tail(n=30)

,Volume_Mn,Value_Cr
92,12020.23,"18,22,949.42"
93,12203.02,"18,41,083.97"
94,12102.67,"18,27,869.35"
95,13440.0,"19,78,353.23"
96,13303.99,"19,64,464.52"
97,14035.84,"20,44,937.05"
98,13885.14,"20,07,081.20"
99,14435.55,"20,64,292.41"
100,14963.05,"20,60,735.57"
101,15041.75,"20,63,994.71"


In [14]:
print(repr(upi['Volume_Mn'].iloc[107:111]))

107    18,301.51\t
108    17,893.42\t
109    18,677.46\t
110    18,395.01\t
Name: Volume_Mn, dtype: object


In [15]:
upi.head(n=25)

,Month,Banks,Volume_Mn,Value_Cr
0,2016-04-01,21,0.0,0.0
1,2016-05-01,21,0.0,0.0
2,2016-06-01,21,0.0,0.0
3,2016-07-01,21,0.09,0.38
4,2016-08-01,21,0.09,3.09
5,2016-09-01,25,0.09,32.64
6,2016-10-01,26,0.1,48.57
7,2016-11-01,30,0.29,100.46
8,2016-12-01,35,1.99,707.93
9,2017-01-01,36,4.46,1696.22


In [16]:
upi['Volume_Mn'].apply(type).value_counts()


<class 'float'>    96
<class 'str'>      26
Name: Volume_Mn, dtype: int64

In [17]:
upi['Value_Cr'].apply(type).value_counts()


<class 'str'>      98
<class 'float'>    24
Name: Value_Cr, dtype: int64

In [18]:
upi['Value_Cr'] = (
    upi['Value_Cr']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('\t', '', regex=False)
)

In [19]:
upi['Volume_Mn'] = (
    upi['Volume_Mn']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('\t', '', regex=False)
)

In [20]:
upi['Value_Cr'] = pd.to_numeric(upi['Value_Cr'], errors='coerce')
upi['Volume_Mn'] = pd.to_numeric(upi['Volume_Mn'], errors='coerce')

In [21]:
upi.head(n=25)

,Month,Banks,Volume_Mn,Value_Cr
0,2016-04-01,21,0.00,0.00
1,2016-05-01,21,0.00,0.00
2,2016-06-01,21,0.00,0.00
3,2016-07-01,21,0.09,0.38
4,2016-08-01,21,0.09,3.09
5,2016-09-01,25,0.09,32.64
6,2016-10-01,26,0.10,48.57
7,2016-11-01,30,0.29,100.46
8,2016-12-01,35,1.99,707.93
9,2017-01-01,36,4.46,1696.22


In [22]:
upi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Month      122 non-null    datetime64[ns]
 1   Banks      122 non-null    int64         
 2   Volume_Mn  122 non-null    float64       
 3   Value_Cr   122 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(1)
memory usage: 3.9 KB


In [23]:
upi.isnull().sum()

Month        0
Banks        0
Volume_Mn    0
Value_Cr     0
dtype: int64

In [24]:
upi.describe()

,Banks,Volume_Mn,Value_Cr
count,122.000000,122.000000,1.220000e+02
mean,303.754098,6352.901557,9.257057e+05
std,229.654369,7157.876742,9.556252e+05
min,21.000000,0.000000,0.000000e+00
25%,128.000000,493.005000,7.679176e+04
50%,222.000000,2686.370000,4.992751e+05
75%,513.250000,11365.415000,1.733748e+06
max,720.000000,23201.930000,2.990424e+06


In [25]:
upi.to_csv("../data/processed/upi_master.csv", index=False)

# Audit Summary

## Dataset Overview

- 122 monthly observations
- Covers UPI transaction activity from [start date] to [end date]

## Data Quality Issues Identified

### DQ-001
Chronological ordering issue after merging yearly files.

Status: Resolved

### DQ-002
Volume and Value columns imported as object datatype.

Status: Resolved

### DQ-003
Measurement units documented.

Status: Resolved

## Conclusion

The dataset has been audited and cleaned and is ready for exploratory analysis.